# QRT-style ML mock 2: market-regime clustering

**Core time:** 45-60 minutes

**Optional extension:** 20-30 minutes
**Goal:** discover and interpret market regimes without target labels.

KNN is a supervised nearest-neighbour method; this assessment uses **KMeans** for clustering and DBSCAN as an optional comparison.

## Scenario and rules

The data contains one row per business day and five market-state features. There is no regime label.

- Never treat `date` as a numeric model feature.
- Impute and scale before distance-based clustering.
- Compare `k=2` through `k=6` using inertia and silhouette score.
- Use PCA only for visualization; fit the final KMeans model on all five scaled features.
- Interpret clusters from original-unit feature profiles, not cluster numbers.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 20261102
FEATURES = ["return_5d", "volatility_20d", "volume_z", "cross_asset_corr_20d", "drawdown_60d"]
ROOT = Path.cwd()
if not (ROOT / "data" / "market_regimes.csv").exists():
    ROOT = ROOT / "QRT_mock_test"
market_data = pd.read_csv(ROOT / "data" / "market_regimes.csv", parse_dates=["date"])
print("shape:", market_data.shape)
display(market_data.head())
display(market_data[FEATURES].describe().T)

def report_check(name, function):
    try:
        function()
        print(f"PASS: {name}")
    except NotImplementedError:
        print(f"NOT YET: {name}")
    except Exception as exc:
        print(f"CHECK: {name} -> {type(exc).__name__}: {exc}")

## Task 1 - Audit, impute, and scale

Implement `prepare_clustering_data`.

- Validate that the required columns exist and dates are unique.
- Return a fitted `Pipeline` containing median imputation and standardization.
- Return the transformed two-dimensional NumPy array in the original row order.

In [ ]:
def prepare_clustering_data(data: pd.DataFrame) -> tuple[Pipeline, np.ndarray]:
    # TODO: fit imputation and scaling on the five numeric features.
    raise NotImplementedError

preprocessor = None
scaled_features = None
try:
    preprocessor, scaled_features = prepare_clustering_data(market_data)
    print("scaled shape:", scaled_features.shape)
    print("largest absolute scaled mean:", np.abs(scaled_features.mean(axis=0)).max())
except NotImplementedError:
    print("Complete prepare_clustering_data, then rerun this cell.")

In [ ]:
def check_clustering_preparation():
    preprocessor, scaled = prepare_clustering_data(market_data)
    assert isinstance(preprocessor, Pipeline)
    assert [name for name, _ in preprocessor.steps] == ["imputer", "scaler"]
    assert scaled.shape == (900, 5)
    assert np.isfinite(scaled).all()
    assert np.abs(scaled.mean(axis=0)).max() < 1e-10

report_check("clustering preparation", check_clustering_preparation)

## Task 2 - Select the number of clusters

Implement `evaluate_kmeans` for every integer `k` from 2 through 6.

Return a DataFrame with exactly `k`, `inertia`, and `silhouette`. Use `n_init=20` and `RANDOM_STATE`. Select the final `k` using the metrics and the interpretability of the profiles.

In [ ]:
def evaluate_kmeans(scaled: np.ndarray, candidates=range(2, 7)) -> pd.DataFrame:
    # TODO: fit each candidate and calculate inertia and silhouette.
    raise NotImplementedError

k_selection = None
try:
    _, scaled_features = prepare_clustering_data(market_data)
    k_selection = evaluate_kmeans(scaled_features)
    display(k_selection)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(k_selection["k"], k_selection["inertia"], marker="o")
    axes[0].set(title="KMeans inertia", xlabel="k", ylabel="inertia")
    axes[1].plot(k_selection["k"], k_selection["silhouette"], marker="o")
    axes[1].set(title="Silhouette score", xlabel="k", ylabel="score")
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Complete evaluate_kmeans, then rerun this cell.")

In [ ]:
def check_k_selection():
    _, scaled = prepare_clustering_data(market_data)
    result = evaluate_kmeans(scaled)
    assert list(result.columns) == ["k", "inertia", "silhouette"]
    assert result["k"].tolist() == [2, 3, 4, 5, 6]
    assert result.loc[result["silhouette"].idxmax(), "k"] == 3
    assert result.loc[result["k"] == 3, "silhouette"].iloc[0] > 0.40

report_check("KMeans model selection", check_k_selection)

## Task 3 - Fit and profile three regimes

Implement:

- `fit_market_clusters`, fitting KMeans with `k=3`, `n_init=20`, and `RANDOM_STATE`, then appending `cluster` to a copy of the raw data;
- `build_cluster_profile`, reporting count, share, and original-unit feature means by cluster.

Cluster IDs are arbitrary. Assign economic names only after inspecting the profile.

In [ ]:
def fit_market_clusters(data: pd.DataFrame, scaled: np.ndarray, n_clusters: int = 3):
    # TODO: return fitted_model, clustered_data.
    raise NotImplementedError

def build_cluster_profile(clustered_data: pd.DataFrame) -> pd.DataFrame:
    # TODO: return count, share, and feature means by cluster.
    raise NotImplementedError

cluster_model = None
clustered_data = None
profile = None
try:
    _, scaled_features = prepare_clustering_data(market_data)
    cluster_model, clustered_data = fit_market_clusters(market_data, scaled_features)
    profile = build_cluster_profile(clustered_data)
    display(profile)
except NotImplementedError:
    print("Complete Task 3, then rerun this cell.")

In [ ]:
def check_cluster_profile():
    _, scaled = prepare_clustering_data(market_data)
    model, clustered = fit_market_clusters(market_data, scaled)
    profile = build_cluster_profile(clustered)
    assert set(clustered["cluster"]) == {0, 1, 2}
    assert profile["count"].sum() == len(market_data)
    assert np.isclose(profile["share"].sum(), 1.0)
    assert profile["share"].min() > 0.10
    assert silhouette_score(scaled, model.labels_) > 0.40

report_check("cluster fit and profile", check_cluster_profile)

## Task 4 - PCA visualization

Implement `make_pca_embedding` to fit a two-component PCA model on the scaled features and return:

1. a DataFrame with `pc1`, `pc2`, and `cluster`;
2. the fitted PCA object.

Plot the observations by cluster. PCA is only an inspection view; do not refit KMeans on the two-dimensional embedding.

In [ ]:
def make_pca_embedding(scaled: np.ndarray, labels: np.ndarray):
    # TODO: return embedding DataFrame and fitted PCA.
    raise NotImplementedError

try:
    _, scaled_features = prepare_clustering_data(market_data)
    cluster_model, clustered_data = fit_market_clusters(market_data, scaled_features)
    embedding, pca = make_pca_embedding(scaled_features, cluster_model.labels_)
    for cluster, group in embedding.groupby("cluster"):
        plt.scatter(group["pc1"], group["pc2"], s=14, alpha=0.55, label=f"cluster {cluster}")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("PCA view of KMeans regimes")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()
    print("explained variance:", pca.explained_variance_ratio_)
except NotImplementedError:
    print("Complete make_pca_embedding, then rerun this cell.")

In [ ]:
def check_pca_embedding():
    _, scaled = prepare_clustering_data(market_data)
    model, _ = fit_market_clusters(market_data, scaled)
    embedding, pca = make_pca_embedding(scaled, model.labels_)
    assert list(embedding.columns) == ["pc1", "pc2", "cluster"]
    assert len(embedding) == len(market_data)
    assert len(pca.explained_variance_ratio_) == 2

report_check("PCA visualization data", check_pca_embedding)

## Optional advanced section

These tasks are not required for the core checks.

1. Refit KMeans with seeds 0 through 10 and calculate adjusted Rand index against seed 0.
2. Fit DBSCAN on the scaled features. Report its number of non-noise clusters, noise fraction, and silhouette excluding noise.
3. Explain why DBSCAN can label observations as noise while KMeans always assigns every observation.

In [ ]:
# OPTIONAL TODO: KMeans stability and DBSCAN comparison.
advanced_clustering_results = None
print("Optional section not run." if advanced_clustering_results is None else advanced_clustering_results)

## Final interpretation

In three or four sentences:

- name the three regimes using their profiles;
- explain why KMeans cluster numbers have no inherent ordering;
- state whether PCA visually supports the segmentation;
- identify one limitation of using spherical KMeans clusters for financial regimes.

**Your answer:** TODO.